# Drift-aware few-shot LLM for evolving SMS spam
Runtime -> Change runtime type -> **T4 GPU**. Run the cells in order.

In [ ]:
!pip -q install transformers accelerate bitsandbytes datasets kagglehub scikit-learn scipy matplotlib

In [ ]:
# Upload the repo zip (drift-fewshot-sms.zip) via the Files panel, then:
!unzip -q -o drift-fewshot-sms.zip -d . 2>/dev/null; %cd drift-fewshot-sms
!python tests/smoke_test.py | tail -3   # mock-model sanity check (no GPU needed)

In [ ]:
# Needed only for Mistral-7B-Instruct-v0.3 (gated): accept terms at huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
from huggingface_hub import notebook_login; notebook_login()

## 1) Base-paper reproduction (SpaLLM-Guard protocol, UCI SMS Spam Collection)

In [ ]:
!python run_reproduction.py --model mistralai/Mistral-7B-Instruct-v0.3

## 2) Main experiments - simulated drift (UCI)

In [ ]:
!python run_drift.py --mode simulated --model Qwen/Qwen2.5-7B-Instruct --max_eval 300
!python make_figures.py --dir results/simulated

## 3) Main experiments - REAL timestamps (optional, needs the Kaggle CSV)
Upload `text-messages.csv` to Colab, or use kagglehub below.

In [ ]:
import kagglehub, glob
path = kagglehub.dataset_download('scottleechua/ph-spam-marketing-sms-w-timestamps')
print(glob.glob(path + '/*'))

In [ ]:
!python run_drift.py --mode real --real_csv {path}/text-messages.csv --model Qwen/Qwen2.5-7B-Instruct --max_eval 300

## 4) Download results

In [ ]:
!zip -q -r results.zip results figures
from google.colab import files; files.download('results.zip')